In [ ]:
from stable_baselines3 import PPO
import supersuit as ss

from multi_car_racing import parallel_env

In [ ]:
env = parallel_env(num_agents=2, render_mode=None, use_ego_color=True)
env = ss.pad_observations_v0(env)
env = ss.pad_action_space_v0(env)
env = ss.dtype_v0(env, 'float32')
env = ss.normalize_obs_v0(env, env_min=0.0, env_max=255.0)
env = ss.frame_stack_v1(env, 4)
env = ss.pettingzoo_env_to_vec_env_v1(env)
env = ss.concat_vec_envs_v1(env, num_vec_envs=4, num_cpus=1, base_class='stable_baselines3')

In [ ]:
model = PPO("MlpPolicy", env, verbose=1, n_steps=128, batch_size=256, n_epochs=4)
model.learn(total_timesteps=10000, progress_bar=True)

In [ ]:
from time import sleep
# from stable_baselines3 import PPO
import supersuit as ss
from multi_car_racing import parallel_env

# 1) create the parallel PettingZoo env with human render
par_env = parallel_env(num_agents=2, render_mode="human", use_ego_color=True, human_show_team_colors=True)

# 2) apply the same wrappers you used for training (order matters)
par_env = ss.pad_observations_v0(par_env)
par_env = ss.pad_action_space_v0(par_env)
par_env = ss.dtype_v0(par_env, "float32")
par_env = ss.normalize_obs_v0(par_env, env_min=0.0, env_max=255.0)
par_env = ss.frame_stack_v1(par_env, 4)

# 3) convert to a single Markov VecEnv (do NOT use multi-cpu concat here)
vec_env = ss.pettingzoo_env_to_vec_env_v1(par_env)

# 4) load the model (or use existing 'model')
# model = PPO.load("models/ppo_latest.zip", env=vec_env)

# 5) run deterministic policy in a loop and render
obs, infos = vec_env.reset()
while True:
    action, _ = model.predict(obs, deterministic=True)
    obs, rewards, terminations, truncations, infos = vec_env.step(action)
    dones = (terminations | truncations)
    vec_env.render()        # will call the env's human renderer
    sleep(0.01)             # adjust to slow down rendering if needed
    if dones.all():         # stop when all agents/envs are done
        break

vec_env.close()